# 03 - Experimentos con MLflow y Optuna

Este notebook permite:
1. Explorar los runs registrados en MLflow
2. Visualizar la convergencia del estudio de Optuna
3. Analizar la importancia de hiperparametros
4. Promover el mejor modelo a Production desde Python

**Prerequisito:** Haber ejecutado el pipeline de entrenamiento:
```bash
uv run python flows/training_pipeline.py
```
o directamente:
```bash
uv run python src/models/train.py
```

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import warnings
warnings.filterwarnings('ignore')

MLFLOW_URI = 'sqlite:///../mlflow.db'
OPTUNA_STORAGE = 'sqlite:///../optuna.db'
EXPERIMENT_NAME = 'diabetes-readmission-prediction'
MODEL_NAME = 'diabetes-readmission-xgboost'

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.tracking.MlflowClient()
print('MLflow conectado en:', MLFLOW_URI)

## 1. Resumen de Todos los Runs en MLflow

In [ ]:
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    print(f'Experimento "{EXPERIMENT_NAME}" no encontrado.')
    print('Ejecuta primero: uv run python src/models/train.py')
else:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.roc_auc DESC'],
        max_results=30,
    )

    results = []
    for run in runs:
        auc = run.data.metrics.get('roc_auc')
        if auc is None:
            continue
        results.append({
            'run_name': run.data.tags.get('mlflow.runName', 'N/A'),
            'model_type': run.data.tags.get('model_type', 'N/A'),
            'roc_auc':   round(auc, 4),
            'recall':    round(run.data.metrics.get('recall', 0), 4),
            'precision': round(run.data.metrics.get('precision', 0), 4),
            'f1':        round(run.data.metrics.get('f1', 0), 4),
            'run_id':    run.info.run_id[:8] + '...',
        })

    results_df = pd.DataFrame(results)
    print(f'Total runs con metricas: {len(results_df)}')
    display(results_df.head(15))

## 2. Visualizacion de Convergencia de Optuna

In [ ]:
import optuna
from pathlib import Path

# Verificar que el archivo de Optuna existe antes de intentar cargar
optuna_path = Path('../optuna.db')

if not optuna_path.exists():
    print('No se encontro optuna.db.')
    print('Ejecuta primero: uv run python flows/training_pipeline.py')
    print('O directamente: uv run python src/models/train.py')
else:
    # Listar estudios disponibles
    available_studies = optuna.study.get_all_study_names(storage=OPTUNA_STORAGE)
    print(f'Estudios disponibles: {available_studies}')

    if not available_studies:
        print('No hay estudios registrados aun. Ejecuta el pipeline de entrenamiento.')
    else:
        # Cargar el estudio mas reciente (o el especifico del proyecto)
        study_name = 'diabetes-readmission-xgb' if 'diabetes-readmission-xgb' in available_studies \
                     else available_studies[0]

        study = optuna.load_study(
            study_name=study_name,
            storage=OPTUNA_STORAGE,
        )

        completed_trials = [t for t in study.trials if t.value is not None]
        print(f'Estudio: {study_name}')
        print(f'Trials completados: {len(completed_trials)}')
        print(f'Mejor AUC: {study.best_value:.4f}')
        print(f'Mejor trial: #{study.best_trial.number}')

In [ ]:
# Graficar solo si el estudio fue cargado exitosamente
if optuna_path.exists() and available_studies:
    trial_values = [t.value for t in completed_trials]
    best_so_far  = [max(trial_values[:i+1]) for i in range(len(trial_values))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel izquierdo: convergencia
    axes[0].plot(trial_values, 'o-', alpha=0.5, color='steelblue', label='Trial AUC')
    axes[0].plot(best_so_far, 'r-', linewidth=2, label='Mejor AUC acumulado')
    axes[0].axhline(y=0.68, color='green', linestyle='--', label='Umbral minimo (0.68)')
    axes[0].set_xlabel('Numero de Trial')
    axes[0].set_ylabel('ROC-AUC')
    axes[0].set_title('Convergencia del Estudio de Optuna')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Panel derecho: importancia de hiperparametros
    try:
        importance = optuna.importance.get_param_importances(study)
        params = list(importance.keys())
        values = list(importance.values())
        axes[1].barh(params, values, color='coral', edgecolor='black')
        axes[1].set_xlabel('Importancia relativa')
        axes[1].set_title('Importancia de Hiperparametros (FAnova)')
    except Exception as e:
        axes[1].text(0.5, 0.5, f'Importancia no disponible\n({e})',
                     ha='center', va='center', transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()

    print('\nMejores hiperparametros encontrados:')
    for k, v in study.best_params.items():
        print(f'  {k}: {round(v, 6) if isinstance(v, float) else v}')

## 3. Promover el Mejor Modelo a Production

In [ ]:
try:
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")

    if not versions:
        print(f'No hay versiones registradas de "{MODEL_NAME}".')
        print('Ejecuta primero el pipeline de entrenamiento.')
    else:
        print(f'Versiones registradas de {MODEL_NAME}:')
        for v in sorted(versions, key=lambda x: int(x.version)):
            print(f'  v{v.version}: stage={v.current_stage} | run={v.run_id[:8]}...')

        # Promover la version mas reciente a Production
        latest = max(versions, key=lambda v: int(v.version))

        print(f'\nPromoviendo v{latest.version} a Production...')
        client.transition_model_version_stage(
            name=MODEL_NAME,
            version=latest.version,
            stage='Production',
            archive_existing_versions=True,
        )
        print(f'Modelo v{latest.version} en Production.')
        print(f'Listo para la API: uv run uvicorn src.api.main:app --reload')

except Exception as e:
    print(f'Error al interactuar con el Model Registry: {e}')

## 4. Verificar la API

Con el modelo en Production, levantar la API en otra terminal:
```bash
uv run uvicorn src.api.main:app --reload --port 8000
```

Verificar salud:
```bash
curl http://localhost:8000/health
```

Documentacion interactiva: http://localhost:8000/docs